In [1]:
import re
import numpy as np
import pandas as pd
from collections import Counter

In [2]:
output__path = f"../Outputs/"

In [3]:
CSV_PATH = f"{output__path}discharge_filtered.csv"
TEXT_COL = "text"

df = pd.read_csv(CSV_PATH)
df = df[df[TEXT_COL].notna()].copy()


SECTION_HEADER_RE = re.compile(r"(?im)^\s*([A-Za-z][A-Za-z0-9 /&\-\(\)\#]+)\s*:\s*$") #It only matches if the “Title:” is alone on the line (because of ^ and $) and uses multi-line mode.
HEADER_BLOCK_RE   = re.compile(r"(?is)\bname:\s*.*?\bchief complaint:\s*") 
#Removes the de-identified “header” part of the note (name/unit/admission date/etc.)
#It deletes everything starting at name: up to the first chief complaint:.


KEEP_PREFIXES = [ #These are sections we want because they often contain meaningful narrative info:
    "chief complaint","history of present illness","brief hospital course",
    "hospital course","hospital course by problem","active issues",
    "chronic issues","transition of care","studies","imaging",
    "other studies","discharge diagnosis","diagnosis",
    "major surgical or invasive procedure",
]

DROP_PREFIXES = [ #These are sections we don’t want because they often create repetitive boilerplate or noise:
    "physical exam","admission physical exam","discharge physical exam",
    "pertinent results","admission labs","discharge labs",
    "interval/discharge labs","micro","pathology",
    "medications on admission","discharge medications",
    "discharge instructions","followup instructions",
    "discharge disposition","facility","discharge condition",
    "social history","family history","review of systems","allergies",
]

def split_sections(text):
    matches = list(SECTION_HEADER_RE.finditer(text))
    if not matches:
        return [("full_text", text)]
    blocks = []
    for i, m in enumerate(matches):
        title = m.group(1).strip().lower() #matched title
        start = m.end() #content: text from end of this header 
        end = matches[i+1].start() if i+1 < len(matches) else len(text) #until the next header
        blocks.append((title, text[start:end].strip())) 
    return blocks #What it returns: A list of (section_title, section_content).

def keep_useful_sections(text):
    blocks = split_sections(text)
    if len(blocks)==1 and blocks[0][0]=="full_text": 
        return text # If section splitting failed (only “full_text”), keep the whole note (fallback).
    kept=[]
    for title,content in blocks:
        if any(title.startswith(d) for d in DROP_PREFIXES):
            continue #If its title starts with something in DROP_PREFIXES → skip it.
        if any(title.startswith(k) for k in KEEP_PREFIXES):
            kept.append(content)#If its title starts with something in KEEP_PREFIXES → keep the content.
    return "\n".join(kept) if kept else text

TEMPLATE_PHRASES = [
    r"\bhistory of present illness\b",
    r"\bbrief hospital course\b",
    r"\bhospital course\b",
    r"\bdischarge diagnosis\b",
    r"\bchief complaint\b",
]

def clean_note(text):
    if not isinstance(text,str):
        return ""
    text = re.sub(HEADER_BLOCK_RE," ",text) # Remove admin header
    text = text.replace("___"," ") #Remove de-identification placeholders
    text = re.sub(r"\[\*\*.*?\*\*\]"," ",text) #Remove de-identification placeholders
    text = keep_useful_sections(text) #text = keep_useful_sections(text)
    text = text.lower()
    for pat in TEMPLATE_PHRASES: #Remove common template phrases
        text = re.sub(pat," ",text)
    text = re.sub(r"=+|_+"," ",text) #Remove divider lines
    text = re.sub(r"\s+"," ",text).strip() #Collapse whitespace
    return text

df["text_clean"] = df[TEXT_COL].apply(clean_note)
#Result: text_clean = cleaned, lowercased narrative text with less boilerplate.

TOKEN_RE = re.compile(r"[a-z]+") #Keeps only sequences of letters (no numbers, no punctuation).

STOPWORDS = set("""
a an the and or but if then else to of in on for with without by from at as is are was were be been being
this that these those it its patient pt she he they them his her their
mg ml mcg hr hrs day days week weeks month months year years yo old female male
had have has not no which who also likely found showed present prior during after
given started continued placed noted received transferred arrival initial report decision made
""".split()) #Removes common English words + common medical-note filler words (pt, mg, day, etc.).

def tokenize(t):
    toks = TOKEN_RE.findall(t)
    return [x for x in toks if x not in STOPWORDS and len(x)>1]
    #Extracts words, Drops stopwords, Drops 1-letter tokens

token_lists = [tokenize(t) for t in df["text_clean"].tolist()] #Creates a list of token lists (one per note).
N = len(token_lists) #number of notes.

def make_ngrams(tokens,n): #Make n-grams from tokens
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def top_ngrams(token_lists, n, topk, min_count):
    """
    Counts occurrences of each n-gram across the dataset (raw frequency).
    Filters out rare ones (min_count).
    Returns the topk most common.
    """
    c=Counter()
    for toks in token_lists:
        if len(toks) >= n:
            c.update(make_ngrams(toks,n))
    items=[(k,v) for k,v in c.items() if v>=min_count]
    items.sort(key=lambda x:x[1],reverse=True)
    return items[:topk]

def doc_freq(token_lists,phrase):
    """
    Counts in how many notes the phrase appears at least once.
    Uses set(...) so multiple repeats in one note count only once.
    """
    n=len(phrase.split())
    cnt=0
    for toks in token_lists:
        if len(toks)>=n and phrase in set(make_ngrams(toks,n)):
            cnt+=1
    return cnt


top_uni = top_ngrams(token_lists,1,50,120)
top_bi  = top_ngrams(token_lists,2,60,40)
top_tri = top_ngrams(token_lists,3,40,25)

print("\nTOP BIGRAMS (doc freq)")
for p,c in top_bi:
    print(f"{doc_freq(token_lists,p):>5}/{N} | {c:>4} | {p}")

#doc_freq/N = number of notes containing it
#c = total raw occurrences across all notes
#p = the phrase

print("\nTOP TRIGRAMS (doc freq)")
for p,c in top_tri:
    print(f"{doc_freq(token_lists,p):>5}/{N} | {c:>4} | {p}")

# =========================
# Indicator dictionary
# =========================
INDICATORS = {
    "Anoxic brain injury": r"\b(anoxic brain|hypoxic ischemic encephal)\w*",
    "Seizure": r"\b(seizure|myoclon)\w*",
    "EEG": r"\b(eeg|electroencephalogram)\b",
    "Coma": r"\b(coma|comatose|unresponsive|unarousable)\b",

    "Vasopressor": r"\b(norepi|levophed|vasopressin|pressor|pressors)\b",
    "Shock": r"\b(cardiogenic shock|septic shock|shock)\b",
    "Hypotension": r"\b(hypotension|sbp\s*<\s*90)\b",

    "TTM": r"\b(ttm|targeted temperature|hypothermia|cooling)\b",
    "ROSC": r"\b(return of spontaneous circulation|rosc)\b",

    "Myocardial infarction": r"\b(stemi|nstemi|myocardial infarct|acute mi)\b",
    "Cardiac cath": r"\b(cardiac cath|catheterization|pci|stent)\b",
    "Arrhythmia": r"\b(vf arrest|vt arrest|arrhythmia|atrial fibrillation)\b",

    "Respiratory failure": r"\b(respiratory failure|hypoxemic respiratory|hypercapnic respiratory)\b",
    "Mechanical ventilation": r"\b(intubated|mechanical ventilation|ventilator)\b",
    "ARDS": r"\b(ards|acute respiratory distress)\b",
    "Pulmonary edema": r"\b(pulmonary edema)\b",

    "AKI": r"\b(aki|acute kidney injury|acute renal failure)\b",
    "Dialysis": r"\b(crrt|dialysis|hemodialysis)\b",

    "Metabolic acidosis": r"\b(metabolic acidosis|lactic acidosis|anion gap)\b",
    "Lactate": r"\b(lactate)\b",

    "Sepsis": r"\b(sepsis|septic)\b",
    "Pneumonia": r"\b(pneumonia|aspiration)\b",

    "DNR/DNI": r"\b(dnr|dni|do not resuscitate|do not intubate)\b",
    "Comfort care": r"\b(comfort measures|cmo|withdrawal of care|palliative)\b",
    "Death": r"\b(expired|pronounced dead|death)\b",
}

# detect
def detect(text,pat):
    """
    Searches case-insensitively in the cleaned text.
    Returns 1 if found, 0 if not.
    """
    return int(bool(re.search(pat,text,flags=re.I)))

for name,pat in INDICATORS.items():#Adds one new column per indicator, containing 0/1 per note.
    df[name] = df["text_clean"].apply(lambda x: detect(x,pat))

rows=[]
for k in INDICATORS:
    cnt=df[k].sum() #cnt = number of notes where it was mentioned (since column is 0/1).
    rows.append((k,cnt,cnt/N)) #cnt/N = fraction of notes mentioning it.

prev=pd.DataFrame(rows,columns=["Indicator","Num_notes","Frac_notes"]).reset_index(drop=True)
prev=prev.sort_values("Num_notes",ascending=False)

print("\nINDICATOR PREVALENCE")
print(prev)


TOP BIGRAMS (doc freq)
  733/1618 | 1754 | cardiac arrest
  556/1618 | 1281 | pea arrest
  507/1618 |  865 | chest pain
  484/1618 |  792 | respiratory failure
  334/1618 |  661 | atrial fibrillation
  317/1618 |  578 | mental status
  333/1618 |  562 | follow up
  199/1618 |  543 | left ventricular
  278/1618 |  495 | heart failure
  293/1618 |  481 | abdominal pain
  231/1618 |  471 | coronary artery
  313/1618 |  419 | ct head
  271/1618 |  414 | pulmonary edema
  241/1618 |  405 | renal failure
  287/1618 |  386 | shortness breath
  279/1618 |  382 | blood pressure
  184/1618 |  381 | aortic valve
  235/1618 |  357 | cardiac catheterization
  235/1618 |  355 | post arrest
  284/1618 |  328 | intubated sedated
  227/1618 |  327 | cooling protocol
  265/1618 |  325 | further management
  245/1618 |  323 | chest compressions
  227/1618 |  311 | head ct
  171/1618 |  309 | brain injury
  239/1618 |  303 | went into
  222/1618 |  301 | line placement
  202/1618 |  301 | pericardial eff

## Detailed Processing:

In [4]:
import re
from typing import Dict, List, Optional


SECTION_PATTERNS = {
    "chief_complaint": [
        r"chief complaint",
        r"reason for admission",
        r"reason for hospitalization",
        r"presenting complaint",
    ],
    "history_present_illness": [
        r"history of present illness",
        r"\bhpi\b",
        r"present illness",
        r"brief hpi",
        r"history of presenting illness",
    ],
    "past_medical_history": [
        r"past medical history",
        r"\bpmh\b",
        r"medical history",
        r"prior medical history",
    ],
    "past_surgical_history": [
        r"past surgical history",
        r"\bpsh\b",
        r"surgical history",
        r"major surgical or invasive procedure",
        r"major surgical or invasive procedures",
    ],
    "physical_exam": [
        r"physical exam",
        r"physical examination",
        r"admission physical exam",
        r"admission pe",
        r"admission exam",
        r"physical examination on admission",
    ],
    "discharge_exam": [
        r"discharge physical exam",
        r"discharge pe",
        r"physical exam on discharge",
        r"physical examination on discharge",
    ],
    "pertinent_results": [
        r"pertinent results",
        r"pertinent laboratory results",
        r"admission labs",
        r"discharge labs",
        r"pertinent labs",
        r"labs on admission",
    ],
    "imaging": [
        r"imaging",
        r"studies",
        r"other studies",
        r"diagnostic studies",
        r"radiology",
    ],
    "hospital_course": [
        r"brief hospital course",
        r"hospital course",
        r"hospital course by problem",
        r"course by problem",
        r"icu course",
        r"brief icu course",
        r"ccu course",
        r"micu course",
        r"sicu course",
        r"tsicu course",
        r"floor course",
    ],
    "transitional_issues": [
        r"transitional issues",
        r"transition of care",
    ],
    "discharge_diagnosis": [
        r"discharge diagnosis",
        r"discharge diagnoses",
        r"final diagnosis",
        r"final diagnoses",
    ],
    "discharge_condition": [
        r"discharge condition",
        r"condition on discharge",
    ],
    "discharge_disposition": [
        r"discharge disposition",
        r"disposition",
        r"facility",
    ],
    "discharge_instructions": [
        r"discharge instructions",
        r"patient instructions",
        r"instructions",
    ],
    "followup": [
        r"followup instructions",
        r"follow-up instructions",
        r"follow up instructions",
        r"followup",
        r"follow-up",
        r"follow up",
    ],
}

MODEL_A_SECTIONS = [
    "chief_complaint",
    "history_present_illness",
    "physical_exam",
    "pertinent_results",
    "imaging",
    "hospital_course",
]

HIGH_LEAKAGE_SECTIONS = [
    "discharge_exam",
    "discharge_diagnosis",
    "discharge_condition",
    "discharge_disposition",
    "discharge_instructions",
    "followup",
    "transitional_issues",
]

LEAKAGE_PATTERNS = [
    r"\bexpired\b",
    r"\bpronounced dead\b",
    r"\bdeath\b",
    r"\bdeceased\b",
    r"\bterminally extubat\w*\b",
    r"\bwithdraw\w* care\b",
    r"\bwithdrawal of care\b",
    r"\bcomfort measures\b",
    r"\bcomfort[- ]oriented care\b",
    r"\bcmo\b",
    r"\bpalliative\b",
    r"\bdnr\b",
    r"\bdni\b",
    r"\bdo not resuscitate\b",
    r"\bdo not intubate\b",
    r"\bgoals of care\b",
    r"\bheroic measures\b",
]

HEADER_BLOCK_RE = re.compile(r"(?is)\bname:\s*.*?\bchief complaint:\s*")


def basic_clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.replace("\r", "\n")
    text = re.sub(HEADER_BLOCK_RE, "Chief Complaint:\n", text)
    text = text.replace("___", " ")
    text = re.sub(r"\[\*\*.*?\*\*\]", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def parse_discharge_sections(note_text: str) -> Dict[str, str]:
    if not isinstance(note_text, str) or not note_text.strip():
        return {}

    text = basic_clean_text(note_text)
    lines = text.split("\n")

    headers = []

    for i, line in enumerate(lines):
        line_clean = line.strip()
        if not line_clean:
            continue

        line_lower = line_clean.lower().rstrip(":")
        for canonical, patterns in SECTION_PATTERNS.items():
            matched = False
            for pat in patterns:
                if re.fullmatch(pat, line_lower, flags=re.I):
                    headers.append((i, canonical, line_clean))
                    matched = True
                    break
            if matched:
                break

    if not headers:
        return {"full_text": text}

    sections = {}
    for idx, (line_no, canonical, _) in enumerate(headers):
        start = line_no + 1
        end = headers[idx + 1][0] if idx + 1 < len(headers) else len(lines)
        content = "\n".join(lines[start:end]).strip()
        if content:
            if canonical in sections:
                sections[canonical] += "\n\n" + content
            else:
                sections[canonical] = content

    return sections


def is_lab_like_line(line: str) -> bool:
    """
    Keep classic lab/result lines intact.
    """
    if not line:
        return False

    patterns = [
        r"\blactate[-:\s]",
        r"\bph[-:\s]",
        r"\bhco3[-:\s]",
        r"\banion gap\b",
        r"\bbase xs\b",
        r"\bglucose[-:\s]",
        r"\bcreat[-:\s]|\bcreatinine\b|\bcreat\b",
        r"\burea\b|\burean\b|\burea n\b",
        r"\bna\+?-",
        r"\bk\+?-|\bpotassium[-:\s]",
        r"\bcl-[-:\s]|\bchloride[-:\s]",
        r"\bblood\b",
        r"\bwbc[-:\s]|\bhgb[-:\s]|\bhct[-:\s]",
        r"\bpo2[-:\s]|\bpco2[-:\s]",
    ]
    return any(re.search(p, line, flags=re.I) for p in patterns)


def is_section_marker_line(line: str) -> bool:
    return bool(re.fullmatch(r"\[[A-Z_]+\]", line.strip()))


def is_problem_header_line(line: str) -> bool:
    """
    Preserve problem-list lines such as:
    # Shock:
    # Metabolic acidosis:
    """
    return bool(re.match(r"^\s*#\s*[^#].*", line))


def normalize_wrapped_lines(text: str) -> str:
    """
    Joins visually wrapped narrative lines while preserving:
    - blank-line paragraph breaks
    - lab/result lines
    - section markers like [PERTINENT_RESULTS]
    - problem headers like # Shock:
    """
    if not text:
        return ""

    raw_lines = [ln.rstrip() for ln in text.split("\n")]
    merged_blocks: List[str] = []
    current = ""

    def flush_current():
        nonlocal current
        if current.strip():
            merged_blocks.append(current.strip())
        current = ""

    for raw in raw_lines:
        line = raw.strip()

        if not line:
            flush_current()
            continue

        if is_section_marker_line(line) or is_problem_header_line(line) or is_lab_like_line(line):
            flush_current()
            merged_blocks.append(line)
            continue

        # start new block if empty
        if not current:
            current = line
            continue

        # if current already ends with sentence punctuation, start new block
        if re.search(r"[.!?:]\s*$", current):
            flush_current()
            current = line
            continue

        # if current looks like a heading, do not merge into it
        if re.fullmatch(r"[A-Z][A-Z /&\-\(\)0-9]+:?", current):
            flush_current()
            current = line
            continue

        # otherwise join wrapped narrative line
        current += " " + line

    flush_current()
    return "\n".join(merged_blocks)


def split_into_sentences_preserve_lines(text: str) -> List[str]:
    """
    First normalize wrapped lines, then split narrative blocks into sentences,
    while preserving lab-like lines and problem headers.
    """
    if not text:
        return []

    text = normalize_wrapped_lines(text)

    out = []
    for line in text.split("\n"):
        line = line.strip()
        if not line:
            continue

        if is_section_marker_line(line) or is_problem_header_line(line) or is_lab_like_line(line):
            out.append(line)
            continue

        parts = re.split(r'(?<=[\.\?\!])\s+', line)
        for p in parts:
            p = p.strip()
            if p:
                out.append(p)

    return out


def remove_leakage_sentences(text: str, leakage_patterns: Optional[List[str]] = None) -> str:
    if not text:
        return ""

    leakage_patterns = leakage_patterns or LEAKAGE_PATTERNS
    sentences = split_into_sentences_preserve_lines(text)

    kept = []
    for sent in sentences:
        sent_lower = sent.lower()
        if any(re.search(pat, sent_lower, flags=re.I) for pat in leakage_patterns):
            continue
        kept.append(sent)

    return "\n".join(kept).strip()


def build_model_a_text(sections: Dict[str, str], strip_leakage_sentences: bool = True) -> str:
    if not sections:
        return ""

    parts = []
    for sec in MODEL_A_SECTIONS:
        if sec in sections and sections[sec].strip():
            content = sections[sec].strip()
            if strip_leakage_sentences:
                content = remove_leakage_sentences(content)
            if content:
                parts.append(f"[{sec.upper()}]\n{content}")

    if parts:
        return "\n\n".join(parts).strip()

    full_text = sections.get("full_text", "")
    if strip_leakage_sentences:
        full_text = remove_leakage_sentences(full_text)
    return full_text.strip()


def filter_sentences_by_keywords(sentences: List[str], keywords: List[str], window: int = 1) -> List[str]:
    if not sentences:
        return []

    keywords_lower = [k.lower() for k in keywords]
    keep_idxs = set()

    for i, sent in enumerate(sentences):
        sent_lower = sent.lower()
        if any(k in sent_lower for k in keywords_lower):
            for j in range(max(0, i - window), min(len(sentences), i + window + 1)):
                keep_idxs.add(j)

    return [sentences[i] for i in sorted(keep_idxs)]


def fallback_regex_lines(text: str, regex_patterns: List[str]) -> List[str]:
    text = normalize_wrapped_lines(text)
    matches = []
    for line in text.split("\n"):
        line_clean = line.strip()
        if not line_clean:
            continue
        if any(re.search(p, line_clean, flags=re.I) for p in regex_patterns):
            matches.append(line_clean)
    return matches


def make_sentence_chunks(sentences: List[str], chunk_size: int = 6, overlap: int = 2) -> List[str]:
    if not sentences:
        return []

    chunks = []
    step = max(1, chunk_size - overlap)

    for start in range(0, len(sentences), step):
        chunk = sentences[start:start + chunk_size]
        if chunk:
            chunks.append("\n".join(chunk))
        if start + chunk_size >= len(sentences):
            break

    return chunks


def prepare_lactate_chunks(
    note_text: str,
    keywords: List[str],
    chunk_size: int = 6,
    overlap: int = 2,
    keyword_window: int = 1,
) -> List[str]:
    if not isinstance(note_text, str) or not note_text.strip():
        return []

    sections = parse_discharge_sections(note_text)
    llm_text = build_model_a_text(sections, strip_leakage_sentences=True)

    if not llm_text:
        return []

    sentences = split_into_sentences_preserve_lines(llm_text)
    filtered = filter_sentences_by_keywords(sentences, keywords, window=keyword_window)

    if not filtered:
        filtered = fallback_regex_lines(
            llm_text,
            regex_patterns=[
                r"\blactate\b",
                r"\blactic acidosis\b",
                r"\bmetabolic acidosis\b",
            ],
        )

    if not filtered:
        return []

    return make_sentence_chunks(filtered, chunk_size=chunk_size, overlap=overlap)


def debug_prepare_lactate(
    note_text: str,
    keywords: List[str],
    chunk_size: int = 6,
    overlap: int = 2,
    keyword_window: int = 1,
) -> Dict[str, object]:
    sections = parse_discharge_sections(note_text)
    llm_text = build_model_a_text(sections, strip_leakage_sentences=True)
    normalized_text = normalize_wrapped_lines(llm_text)
    sentences = split_into_sentences_preserve_lines(llm_text)
    filtered = filter_sentences_by_keywords(sentences, keywords, window=keyword_window)

    if not filtered:
        filtered = fallback_regex_lines(
            llm_text,
            regex_patterns=[
                r"\blactate\b",
                r"\blactic acidosis\b",
                r"\bmetabolic acidosis\b",
            ],
        )

    chunks = make_sentence_chunks(filtered, chunk_size=chunk_size, overlap=overlap)

    return {
        "sections": sections,
        "llm_text": llm_text,
        "normalized_text": normalized_text,
        "sentences": sentences,
        "filtered_sentences": filtered,
        "chunks": chunks,
    }

In [5]:
LACTATE_KEYWORDS = [
    "lactate",
    "lactic acidosis",
    "metabolic acidosis",
]

result = debug_prepare_lactate(df.text[200], LACTATE_KEYWORDS)

## Simple Processing

In [6]:
import re
from typing import List


def clean_raw_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.replace("\r", "\n")
    text = text.replace("___", " ")
    text = re.sub(r"\[\*\*.*?\*\*\]", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def is_lab_like_line(line: str) -> bool:
    return bool(re.search(
        r"\b(lactate|glucose|ph|hco3|anion gap|base xs|wbc|hgb|hct|blood)\b",
        line,
        flags=re.I
    ))


def normalize_wrapped_lines(text: str) -> str:
    """
    Join wrapped narrative lines, but keep lab-like lines separate.
    """
    if not text:
        return ""

    lines = [line.strip() for line in text.split("\n")]
    merged = []
    current = ""

    def flush():
        nonlocal current
        if current.strip():
            merged.append(current.strip())
        current = ""

    for line in lines:
        if not line:
            flush()
            continue

        if is_lab_like_line(line):
            flush()
            merged.append(line)
            continue

        if not current:
            current = line
        elif re.search(r"[.!?:]\s*$", current):
            flush()
            current = line
        else:
            current += " " + line

    flush()
    return "\n".join(merged)


def split_into_sentences(text: str) -> List[str]:
    if not text:
        return []

    text = normalize_wrapped_lines(text)
    sentences = []

    for line in text.split("\n"):
        line = line.strip()
        if not line:
            continue

        if is_lab_like_line(line):
            sentences.append(line)
            continue

        parts = re.split(r'(?<=[\.\?\!])\s+', line)
        for p in parts:
            p = p.strip()
            if p:
                sentences.append(p)

    return sentences


def select_keyword_sentences(
    text: str,
    keywords: List[str],
    window: int = 0,
) -> List[str]:
    text = clean_raw_text(text)
    sentences = split_into_sentences(text)

    if not sentences:
        return []

    keywords_lower = [k.lower() for k in keywords]
    keep_idxs = set()

    for i, sent in enumerate(sentences):
        sent_lower = sent.lower()
        if any(k in sent_lower for k in keywords_lower):
            for j in range(max(0, i - window), min(len(sentences), i + window + 1)):
                keep_idxs.add(j)

    return [sentences[i] for i in sorted(keep_idxs)]


def make_chunks(sentences: List[str], chunk_size: int = 5, overlap: int = 1) -> List[str]:
    if not sentences:
        return []

    chunks = []
    step = max(1, chunk_size - overlap)

    for start in range(0, len(sentences), step):
        chunk = sentences[start:start + chunk_size]
        if chunk:
            chunks.append(" ".join(chunk))
        if start + chunk_size >= len(sentences):
            break

    return chunks


def prepare_lactate_chunks(
    note_text: str,
    keywords: List[str],
    window: int = 0,
    chunk_size: int = 5,
    overlap: int = 1,
) -> List[str]:
    selected_sentences = select_keyword_sentences(
        text=note_text,
        keywords=keywords,
        window=window,
    )

    return make_chunks(
        sentences=selected_sentences,
        chunk_size=chunk_size,
        overlap=overlap,
    )

In [7]:
prepare_lactate_chunks(df.text[200], LACTATE_KEYWORDS)

['01:10PM GLUCOSE-122* LACTATE-1.9 NA+-139 K+-6.2* She was initiated on CRRT for correction of her hyperkalemia and severe metabolic acidosis. She had normal lactates on presentation and on repeat checks. CRRT was done as above to correct metabolic acidosis. # Metabolic acidosis: anion gap of 17, likely from renal failure']

In [8]:
df.columns

Index(['note_id', 'subject_id', 'hadm_id', 'note_type', 'note_seq',
       'charttime', 'storetime', 'text', 'text_clean', 'Anoxic brain injury',
       'Seizure', 'EEG', 'Coma', 'Vasopressor', 'Shock', 'Hypotension', 'TTM',
       'ROSC', 'Myocardial infarction', 'Cardiac cath', 'Arrhythmia',
       'Respiratory failure', 'Mechanical ventilation', 'ARDS',
       'Pulmonary edema', 'AKI', 'Dialysis', 'Metabolic acidosis', 'Lactate',
       'Sepsis', 'Pneumonia', 'DNR/DNI', 'Comfort care', 'Death'],
      dtype='object')

In [9]:
df

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text,text_clean,Anoxic brain injury,...,Pulmonary edema,AKI,Dialysis,Metabolic acidosis,Lactate,Sepsis,Pneumonia,DNR/DNI,Comfort care,Death
0,10001884-DS-38,10001884,26184834,DS,38,2131-01-20 00:00:00,2131-01-20 09:41:00,\nName: ___ Unit No: ___\n \nA...,intubation & mechanical ventilation (extubated...,0,...,0,0,0,0,1,0,1,1,0,0
1,10010471-DS-22,10010471,29842315,DS,22,2155-12-07 00:00:00,2155-12-09 10:22:00,\nName: ___ Unit No: ___\n ...,": rhc y/o f with h/o severe aortic stenosis, e...",0,...,0,0,1,0,1,0,0,0,1,0
2,10013569-DS-10,10013569,27993048,DS,10,2167-12-25 00:00:00,2167-12-25 16:07:00,\nName: ___ Unit No: ___\n...,r thoracentesis right heart cath x2 swan place...,0,...,1,0,0,0,0,0,1,0,1,0
3,10024982-DS-20,10024982,25154057,DS,20,2203-10-11 00:00:00,2203-10-12 00:21:00,\nName: ___ Unit No: ___...,cardiac catherization l3-l4 arterial embolizat...,0,...,1,0,0,0,0,0,1,0,0,0
4,10026161-DS-7,10026161,24614671,DS,7,2133-11-15 00:00:00,2133-11-18 20:57:00,\nName: ___ Unit No: ___\n ...,-proctocolectomy -tracheal intubation -cardiac...,0,...,0,0,0,0,1,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1613,19969137-DS-15,19969137,20917922,DS,15,2143-03-31 00:00:00,2143-04-02 13:46:00,\nName: ___ Unit No: ___\n \n...,intubated/sedated for mra head and neck and li...,0,...,1,0,0,0,1,1,0,0,0,0
1614,19975796-DS-6,19975796,25848942,DS,6,2148-12-14 00:00:00,2148-12-15 13:59:00,\nName: ___ Unit No: ___\...,intubation and extubation central line placeme...,0,...,0,0,0,0,1,1,1,0,0,0
1615,19983257-DS-23,19983257,21588174,DS,23,2166-06-19 00:00:00,2166-06-19 22:04:00,\nName: ___ Unit No: __...,esophagogastroduodenoscopy colonoscopy y/o m w...,0,...,0,0,0,0,0,1,0,0,0,1
1616,19990427-DS-3,19990427,29695607,DS,3,2182-01-24 00:00:00,2182-01-26 07:07:00,\nName: ___ Unit No: __...,anterior lumbar decompression and fusion l3-s1...,1,...,0,0,0,0,0,1,0,0,1,0


In [14]:
df.text[500]

" \nName:  ___           Unit No:   ___\n \nAdmission Date:  ___              Discharge Date:   ___\n \nDate of Birth:  ___             Sex:   F\n \nService: MEDICINE\n \nAllergies: \nAspirin / Bactrim\n \nAttending: ___\n \n___ Complaint:\nfatigue, enlarged lymph nodes\n \nMajor Surgical or Invasive Procedure:\nlymph node biopsy\n \nHistory of Present Illness:\nThis is a ___ with a history of\nHIV (CD4 count 337 in ___ on HAART, reported G6PD deficiency\n(see below), depression and essential tremor who presents to the\nED with fatigue and lymphadenopathy concerning for lymphoma.\n\nShe reports first developing a mass in her left axilla about two\nweeks ago. She had some nausea, vomiting and ? fevers initially,\nalthough these then resolved. She then developed more swelling \nin\nher neck and groin, and these areas were tender and growing. She\nnotes about a 20lb weight loss over three months, with about 15\nlbs in the last ___ weeks. Over the past few days, she has also\nnoticed extre

In [15]:
import random

In [43]:
np.floor(random.uniform(0,1619))

1437.0

In [47]:
for i in range(0, 10):
    random_num = np.floor(random.uniform(0,1619))
    print('index: ', random_num)
    display(df.text[np.floor(random.uniform(0,1619))])
    print('**********************************************')

index:  1267.0


" \nName:  ___                Unit No:   ___\n \nAdmission Date:  ___              Discharge Date:   ___\n \nDate of Birth:  ___             Sex:   F\n \nService: MEDICINE\n \nAllergies: \nCefepime / Meropenem / Ciprofloxacin / Levofloxacin\n \nAttending: ___\n \nChief Complaint:\nFebrile neutropenia\n \nMajor Surgical or Invasive Procedure:\nnone\n \nHistory of Present Illness:\n___ is a ___ female with relapsed AML who \nwas only recently azacitidine was last admitted on ___ with \nneutropenic fever. During this admission she was noted to have \npseudomonas bacteremia as well as klebsiella and ecoli in her \nBAL. She was initially treated with empiric therapy of \nvancomycin and zosyn. Voriconazole was also started initially \nbut this was scaled back after BAl was negative for fungal \ninfection. She was then discharged home on po cefpodoxime after \nshe decided to go home so she can focus on quality of life \ninstead. Patient now presents again with fever for two days \nalong with 

**********************************************
index:  835.0


" \nName:  ___            Unit No:   ___\n \nAdmission Date:  ___              Discharge Date:   ___\n \nDate of Birth:  ___             Sex:   F\n \nService: MEDICINE\n \nAllergies: \nlisinopril\n \nAttending: ___.\n \nChief Complaint:\nLeft Lung Collapse\n \nMajor Surgical or Invasive Procedure:\nBronchoscopy ___\n \nHistory of Present Illness:\n___ year old ___ female with a past medical history of mild \ndementia, ___, AF not on anticoagulation, and DM who presented \nfor evaluation of likely foreign body aspiration. On \npresentation to OSH (___), CXR showed complete white out \nof the left side. CT angiogram showed no PE, but occlusion of \nthe left mainstem bronchus. Pulmonary did flex bronch on ___ \n___, and they felt that she had foreign body, but they did not \nthink they could intervene on it. She has been seen by \nCardiology at ___ for tachycardia in 110s; they felt that her \ntachycardia was caused by her lung collapse. OSH spoke with IP \nat ___ on ___ who said they may

**********************************************
index:  688.0


...

**********************************************
index:  830.0


" \nName:  ___                 Unit No:   ___\n \nAdmission Date:  ___              Discharge Date:   ___\n \nDate of Birth:  ___             Sex:   F\n \nService: MEDICINE\n \nAllergies: \nNo Known Allergies / Adverse Drug Reactions\n \nAttending: ___\n \nChief Complaint:\nPrimary Oncologist:  ___\n.  \nPrimary Care Physician: ___\n.\nChief Complaint:  fatigue, weak without appetite\n\n \nMajor Surgical or Invasive Procedure:\ninternal cardiac defibrillator implantation\nTemporary pacer wire implantation\n\n \nHistory of Present Illness:\n___ with history of high grade, large urothelial bladder cancer \nwith invasion s/p resection ___, HTN, GERD, anxiety and \ndepression referred in from ___ clinic due to elevated WBC.\n.\nThe patient presented to clinic yesterday where she was noted to \nhave an elevated WBC to 12.6 with left shift (89% polys). She \nwas referred in for evaluation, however she declined because she \nwas feeling well at the time. This AM however, she awoke feeling \nw

**********************************************
index:  1300.0


" \nName:  ___                    Unit No:   ___\n \nAdmission Date:  ___              Discharge Date:   ___\n \nDate of Birth:  ___             Sex:   M\n \nService: MEDICINE\n \nAllergies: \nNo Known Allergies / Adverse Drug Reactions\n \nAttending: ___.\n \nChief Complaint:\nChief Complaint: Cardiac arrest\nReason for MICU transfer: Intubated, cooling protocol\n \nMajor Surgical or Invasive Procedure:\nIntubation ___\n \nHistory of Present Illness:\n___ M h/o heroin abuse presents as transfer from OSH intubated \nafter cardiac arrest.  He was released from the department of \ncorrections today and went to his parents' bathroom to take a \nshower.  He was ___ the bathroom for ___ minutes when his \nfather went to check on him and he had not made it to the \nshower, was slumped against the bathroom wall and unresponsive.  \nHe was pulseless with agonal breathing and a respiratory rate of \n2 per minute.  He received 2mg Narcan and 3mg epi and was \nintubated ___ the field.  He was ___

**********************************************


In [52]:
CSV_PATH = f"{output__path}discharge_filtered.csv"
TEXT_COL = "text"

radiology_path = f"{output__path}radiology_earliest.csv"

df = pd.read_csv(CSV_PATH)
df_rad = pd.read_csv(radiology_path)

In [58]:
df_rad['text'][800]

'INDICATION:  ___ female with fall.\n\nCOMPARISON:  Outside hospital radiographs dated ___ at 22:28.\n\nRIGHT FEMUR, THREE VIEWS:  The knee is held in persistent flexion.  There is a\ncomminuted acute fracture of the distal femoral metadiaphysis, with posterior\ndisplacement of the distal fracture fragments by nearly one complete shaft\nwidth.  There is an associated joint effusion.  The patella and proximal tibia\nand fibula appear intact.  There is no evidence of intra-articular extension.\n\nIMPRESSION:  Comminuted fracture of distal femur, with posterior displacement\nof distal fragments, and associated joint effusion.\n'

In [54]:
df

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
0,10001884-DS-38,10001884,26184834,DS,38,2131-01-20 00:00:00,2131-01-20 09:41:00,\nName: ___ Unit No: ___\n \nA...
1,10010471-DS-22,10010471,29842315,DS,22,2155-12-07 00:00:00,2155-12-09 10:22:00,\nName: ___ Unit No: ___\n ...
2,10013569-DS-10,10013569,27993048,DS,10,2167-12-25 00:00:00,2167-12-25 16:07:00,\nName: ___ Unit No: ___\n...
3,10024982-DS-20,10024982,25154057,DS,20,2203-10-11 00:00:00,2203-10-12 00:21:00,\nName: ___ Unit No: ___...
4,10026161-DS-7,10026161,24614671,DS,7,2133-11-15 00:00:00,2133-11-18 20:57:00,\nName: ___ Unit No: ___\n ...
...,...,...,...,...,...,...,...,...
1613,19969137-DS-15,19969137,20917922,DS,15,2143-03-31 00:00:00,2143-04-02 13:46:00,\nName: ___ Unit No: ___\n \n...
1614,19975796-DS-6,19975796,25848942,DS,6,2148-12-14 00:00:00,2148-12-15 13:59:00,\nName: ___ Unit No: ___\...
1615,19983257-DS-23,19983257,21588174,DS,23,2166-06-19 00:00:00,2166-06-19 22:04:00,\nName: ___ Unit No: __...
1616,19990427-DS-3,19990427,29695607,DS,3,2182-01-24 00:00:00,2182-01-26 07:07:00,\nName: ___ Unit No: __...
